In [6]:
!pip install datasets lxml cairosvg sentencepiece tokenizers tqdm matplotlib

^C


^C


^C



[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# -*- coding: utf-8 -*-
"""
Part 4: Best Model Training and Sample Generation

This script trains a transformer on SVG token sequences, evaluates it, generates
unconditional and prefix-conditioned SVG samples, and computes XML/rendering
validity metrics.

Usage:
  python part4.py
  python part4.py --model xl --epochs 4 --batch-size 8
"""

import json
import math
import os
import random
import sys
import time
from dataclasses import dataclass, field
from pathlib import Path

#import cairosvg
import torch
import torch.nn as nn
import torch.optim as optim
from lxml import etree
from matplotlib import pyplot as plt
from tokenizers import Tokenizer
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from typing import Optional

# Paths and defaults
TOKENIZER_PATH = Path('svg_tokenizer_6.json')
TRAIN_TOKENS_PATH = Path('train_tokens_6.pt')
VAL_TOKENS_PATH = Path('val_tokens_6.pt')
TEST_TOKENS_PATH = Path('test_tokens_6.pt')
PART3_LR_PATH = Path('part3_mup_results/best_lr.txt')
PART3_RESULTS_PATH = Path('part3_mup_results/part3_results.pt')
PART2_RESULTS_DIR = Path('part2_scaling_results')
OUTPUT_DIR = Path('part4_outputs')
SAMPLES_DIR = OUTPUT_DIR / 'samples'
RENDER_DIR = OUTPUT_DIR / 'rendered'
METRICS_PATH = OUTPUT_DIR / 'metrics.json'
MODEL_PATH = OUTPUT_DIR / 'best_model.pt'

os.makedirs(SAMPLES_DIR, exist_ok=True)
os.makedirs(RENDER_DIR, exist_ok=True)

@dataclass
class Config:
    model: str = 'auto'
    epochs: int = 3
    batch_size: int = 32
    block_size: int = 512
    lr: Optional[float] = None
    dropout: float = 0.1
    weight_decay: float = 0.01
    max_gen_tokens: int = 256
    top_k: int = 40
    top_p: float = 0.9
    sample_temps: list[float] = field(default_factory=lambda: [0.5, 0.8, 1.0])
    eval_steps: int = 20
    eval_batch_size: int = 8
    log_interval: int = 50
    no_train: bool = False

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=1024):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        position = torch.arange(max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        seq_len = x.size(1)
        x = x + self.pe[:seq_len, :].unsqueeze(0)
        return self.dropout(x)


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.fc_out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        batch, seq_len, _ = query.size()
        Q = self.q_linear(query).view(batch, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        K = self.k_linear(key).view(batch, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        V = self.v_linear(value).view(batch, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        scale = math.sqrt(self.head_dim)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / scale
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(self.dropout(attn), V)
        out = out.transpose(1, 2).contiguous().view(batch, seq_len, self.d_model)
        return self.fc_out(out)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        attn = self.attention(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn))
        f = self.ff(x)
        x = self.norm2(x + self.dropout(f))
        return x


class DecoderOnlyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, dropout=0.1, max_len=1024, padding_idx=None):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=padding_idx)
        self.pos_enc = PositionalEncoding(d_model, dropout, max_len)
        self.initial_ln = nn.LayerNorm(d_model)
        self.layers = nn.ModuleList([TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.final_ln = nn.LayerNorm(d_model)
        self.fc_out = nn.Linear(d_model, vocab_size)

    def _generate_mask(self, sz, device):
        mask = torch.triu(torch.ones(sz, sz, device=device), diagonal=1).bool()
        return ~mask

    def forward(self, src):
        mask = self._generate_mask(src.size(1), src.device)
        x = self.token_embedding(src)
        x = self.pos_enc(x * math.sqrt(self.token_embedding.embedding_dim))
        x = self.initial_ln(x)
        for layer in self.layers:
            x = layer(x, mask)
        x = self.final_ln(x)
        return self.fc_out(x)


class TokenDataset:
    def __init__(self, sequences):
        self.data = sequences

    def sample_batch(self, batch_size, block_size, device):
        x_batch = []
        y_batch = []
        while len(x_batch) < batch_size:
            seq = random.choice(self.data)
            if len(seq) < block_size + 1:
                continue
            start = random.randint(0, len(seq) - block_size - 1)
            chunk = seq[start:start + block_size + 1]
            x_batch.append(torch.tensor(chunk[:-1], dtype=torch.long))
            y_batch.append(torch.tensor(chunk[1:], dtype=torch.long))
        x = torch.stack(x_batch).to(device)
        y = torch.stack(y_batch).to(device)
        return x, y


def get_scheduler(optimizer, total_steps, warmup_steps):
    warmup = LinearLR(optimizer, start_factor=1e-6, end_factor=1.0, total_iters=max(1, warmup_steps))
    cosine = CosineAnnealingLR(optimizer, T_max=max(1, total_steps - warmup_steps))
    return SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_steps])


def steps_per_epoch(total_tokens, batch_size, block_size):
    tokens_per_step = batch_size * block_size
    return max(1, math.ceil(total_tokens / tokens_per_step))


def train_one_epoch(model, dataset, optimizer, scheduler, device, steps, batch_size, block_size, log_interval=100):
    model.train()
    criterion = nn.CrossEntropyLoss()
    running = 0.0
    recorded = []
    start = time.time()
    for step in range(1, steps + 1):
        x, y = dataset.sample_batch(batch_size, block_size, device)
        out = model(x)
        loss = criterion(out.view(-1, out.size(-1)), y.view(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        running += loss.item()
        if step % log_interval == 0 or step == steps:
            avg = running / min(log_interval, step)
            recorded.append(avg)
            print(f"Step {step}/{steps} avg_loss={avg:.4f} lr={scheduler.get_last_lr()[0]:.6g}")
            running = 0.0
    elapsed = time.time() - start
    throughput = steps * batch_size * block_size / max(1e-9, elapsed)
    return recorded, elapsed, throughput


def evaluate(model, dataset, device, steps=100, batch_size=8, block_size=512):
    model.eval()
    crit = nn.CrossEntropyLoss()
    total_loss = 0.0
    with torch.no_grad():
        for _ in range(steps):
            x, y = dataset.sample_batch(batch_size, block_size, device)
            out = model(x)
            loss = crit(out.view(-1, out.size(-1)), y.view(-1))
            total_loss += loss.item()
    return total_loss / steps


def top_k_top_p_filtering(logits, top_k=0, top_p=0.0):
    if top_k > 0:
        indices_to_remove = logits < torch.topk(logits, top_k)[0][..., -1, None]
        logits = logits.masked_fill(indices_to_remove, float('-inf'))
    if top_p > 0.0:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        cumulative_probs = torch.softmax(sorted_logits, dim=-1).cumsum(dim=-1)
        sorted_indices_to_remove = cumulative_probs > top_p
        sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
        sorted_indices_to_remove[..., 0] = False
        indices_to_remove = sorted_indices[sorted_indices_to_remove]
        logits[..., indices_to_remove] = float('-inf')
    return logits


def sample_sequence(model, tokenizer, prefix_ids, max_new_tokens, device, temperature=1.0, top_k=0, top_p=0.0):
    model.eval()
    generated = prefix_ids[:]
    with torch.no_grad():
        for _ in range(max_new_tokens):
            context = torch.tensor([generated[-512:]], dtype=torch.long, device=device)
            logits = model(context)
            next_token_logits = logits[0, -1, :] / max(1e-8, temperature)
            next_token_logits = top_k_top_p_filtering(next_token_logits, top_k=top_k, top_p=top_p)
            probs = torch.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()
            generated.append(next_token)
            if next_token == tokenizer.token_to_id('</s>') if hasattr(tokenizer, 'token_to_id') else False:
                break
    return generated


def decode_tokens(tokenizer, token_ids):
    try:
        return tokenizer.decode(token_ids)
    except Exception:
        return ''.join(tokenizer.id_to_token(tok) for tok in token_ids)


def xml_valid(svg_text):
    try:
        root = etree.fromstring(svg_text.encode('utf-8'))
        tag = root.tag
        return tag.endswith('svg') or tag.endswith('}svg')
    except Exception:
        return False


def render_svg(svg_text, output_path):
    try:
        #cairosvg.svg2png(bytestring=svg_text.encode('utf-8'), write_to=str(output_path), output_width=256, output_height=256)
        return True
    except Exception:
        return False


def safe_encode(tokenizer, text):
    if hasattr(tokenizer, 'encode'):
        return tokenizer.encode(text).ids
    if hasattr(tokenizer, 'encode_batch'):
        return tokenizer.encode_batch([text])[0].ids
    raise RuntimeError('Unsupported tokenizer type')


def load_best_lr():
    if PART3_LR_PATH.exists():
        try:
            return float(PART3_LR_PATH.read_text().strip())
        except Exception:
            pass
    return 2e-4


def load_best_model_from_results():
    candidates = []
    if PART3_RESULTS_PATH.exists():
        try:
            data = torch.load(PART3_RESULTS_PATH,weights_only=False)
            for entry in data.get('mup_results', []):
                if 'name' in entry and 'val_loss' in entry:
                    candidates.append((entry['name'], entry['val_loss']))
        except Exception:
            pass
    if not candidates:
        return None
    best_name, _ = min(candidates, key=lambda x: x[1])
    return best_name


def plot_sample_grid(png_paths, filename, ncols=4):
    n = len(png_paths)
    ncols = min(ncols, n)
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(3 * ncols, 3 * nrows))
    if nrows == 1 and ncols == 1:
        axes = [[axes]]
    elif nrows == 1:
        axes = [axes]
    elif ncols == 1:
        axes = [[ax] for ax in axes]
    for i, ax in enumerate(sum(map(list, axes), [])):
        ax.axis('off')
        if i < n:
            img = plt.imread(png_paths[i])
            ax.imshow(img)
    plt.tight_layout()
    fig.savefig(filename, dpi=200)
    plt.close(fig)



In [ ]:
if not TOKENIZER_PATH.exists() or not TRAIN_TOKENS_PATH.exists() or not VAL_TOKENS_PATH.exists():
    raise FileNotFoundError('Required files missing: svg_tokenizer_6.json, train_tokens_6.pt, val_tokens_6.pt')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
train_tokens = torch.load(str(TRAIN_TOKENS_PATH))
val_tokens = torch.load(str(VAL_TOKENS_PATH))
test_tokens = None
if TEST_TOKENS_PATH.exists():
    test_tokens = torch.load(str(TEST_TOKENS_PATH))

total_train_tokens = sum(len(seq) for seq in train_tokens)
print(f'Train sequences: {len(train_tokens)}, total tokens: {total_train_tokens}')
print(f'Validation sequences: {len(val_tokens)}')
if test_tokens is not None:
    print(f'Test sequences: {len(test_tokens)}')

train_ds = TokenDataset(train_tokens)
val_ds = TokenDataset(val_tokens)
test_ds = TokenDataset(test_tokens) if test_tokens is not None else None

model_configs = {
    'tiny': {'d_model': 128, 'n_layers': 6, 'n_heads': 4, 'd_ff': 512},
    'small': {'d_model': 192, 'n_layers': 6, 'n_heads': 6, 'd_ff': 768},
    'medium': {'d_model': 384, 'n_layers': 6, 'n_heads': 12, 'd_ff': 1536},
    'large': {'d_model': 512, 'n_layers': 6, 'n_heads': 16, 'd_ff': 2048},
    'xl': {'d_model': 768, 'n_layers': 6, 'n_heads': 24, 'd_ff': 3072},
}

Using device: cpu


Using device: cpu


KeyboardInterrupt: 

In [ ]:
data = torch.load(PART3_RESULTS_PATH,weights_only=False)
print(data)

{'sp_results': [{'name': 'tiny', 'params': 2.3356, 'val_loss': 0.7920647835731507, 'train_loss_last': 0.41848366260528563, 'epoch_time': 977.719162940979, 'throughput': 123133.14146146836, 'peak_memory_gb': 4.1182026863098145, 'train_curve': [7.821863446235657, 4.8303602194786075, 2.438209155797958, 1.897413489818573, 1.7888004958629609, 1.7194562947750092, 1.6559951519966125, 1.6145192432403563, 1.5635141229629517, 1.5222461187839509, 1.485230597257614, 1.4378235673904418, 1.3835630548000335, 1.3398193180561067, 1.3168756592273712, 1.272099802494049, 1.237872976064682, 1.208385398387909, 1.175271143913269, 1.1479861974716186, 1.1302043223381042, 1.109883930683136, 1.10500412940979, 1.0817075222730637, 1.066507853269577, 1.0573011088371276, 1.0396343809366226, 1.035825019478798, 1.0217306315898895, 1.0210417026281358, 1.0106244421005248, 0.9987491685152053, 0.9940202671289444, 0.9769272625446319, 0.9766888320446014, 0.9790509158372879, 0.9645217615365982, 0.9564457279443741, 0.94640789

In [ ]:
def main(args: Config):
    if args.model == 'auto':
        chosen = load_best_model_from_results()
        if chosen is not None:
            print('Auto-selected best model from Part 2/Part 3 results:', chosen)
            args.model = chosen
        else:
            args.model = 'large'
            print('No result files found; falling back to default model:', args.model)

    if args.model not in model_configs:
        raise ValueError(f'Model {args.model} is not one of {list(model_configs.keys())}')

    cfg = model_configs[args.model]
    vocab_size = tokenizer.get_vocab_size()
    model = DecoderOnlyTransformer(
        vocab_size=vocab_size,
        d_model=cfg['d_model'],
        n_layers=cfg['n_layers'],
        n_heads=cfg['n_heads'],
        d_ff=cfg['d_ff'],
        dropout=args.dropout,
        max_len=args.block_size,
        padding_idx=None,
    ).to(device)

    print('Model configuration:', args.model, cfg)
    print('Total parameters: {:.2f}M'.format(sum(p.numel() for p in model.parameters()) / 1e6))

    lr = args.lr if args.lr is not None else load_best_lr()
    if args.model in ['large', 'xl']:
        lr *= 0.6
    print('Using learning rate:', lr)

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=args.weight_decay)
    epoch_steps = steps_per_epoch(total_train_tokens, args.batch_size, args.block_size)
    warmup_steps = max(1, epoch_steps // 5)
    scheduler = get_scheduler(optimizer, epoch_steps * args.epochs, warmup_steps)

    if args.no_train:
        if MODEL_PATH.exists():
            print('Loading pretrained model from', MODEL_PATH)
            model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
        else:
            raise FileNotFoundError('Model checkpoint not found. Remove --no-train to train a model.')
    else:
        print(f'Training for {args.epochs} epochs, {epoch_steps} steps per epoch...')
        for epoch in range(1, args.epochs + 1):
            train_loss, elapsed, throughput = train_one_epoch(
                model, train_ds, optimizer, scheduler, device,
                epoch_steps, args.batch_size, args.block_size,
                log_interval=args.log_interval,
            )
            val_loss = evaluate(model, val_ds, device, steps=args.eval_steps, batch_size=args.eval_batch_size, block_size=args.block_size)
            print(f'Epoch {epoch} finished: train_last={train_loss[-1]:.4f} val_loss={val_loss:.4f} elapsed={elapsed:.1f}s throughput={throughput:.1f} tok/s')
        torch.save(model.state_dict(), MODEL_PATH)
        print('Saved trained model to', MODEL_PATH)

    eval_loss = evaluate(model, val_ds, device, steps=args.eval_steps, batch_size=args.eval_batch_size, block_size=args.block_size)
    eval_perplexity = math.exp(eval_loss)
    print(f'Validation loss: {eval_loss:.4f}, perplexity: {eval_perplexity:.4f}')

    if test_ds is not None:
        test_loss = evaluate(model, test_ds, device, steps=args.eval_steps, batch_size=args.eval_batch_size, block_size=args.block_size)
        test_perplexity = math.exp(test_loss)
        print(f'Test loss: {test_loss:.4f}, perplexity: {test_perplexity:.4f}')
    else:
        test_loss = None
        test_perplexity = None

    unconditional_prefix = '<svg '
    unconditional_samples = []
    print('Generating unconditional samples...')
    for idx in range(10):
        temp = args.sample_temps[idx % len(args.sample_temps)]
        generated_ids = sample_sequence(
            model,
            tokenizer,
            safe_encode(tokenizer, unconditional_prefix),
            max_new_tokens=args.max_gen_tokens,
            device=device,
            temperature=temp,
            top_k=args.top_k,
            top_p=args.top_p,
        )
        svg_text = decode_tokens(tokenizer, generated_ids)
        svg_name = f'uncond_{idx+1:02d}_t{temp:.1f}.svg'
        svg_path = SAMPLES_DIR / svg_name
        svg_path.write_text(svg_text, encoding='utf-8')
        png_path = RENDER_DIR / svg_name.replace('.svg', '.png')
        rendered = render_svg(svg_text, png_path)
        valid_xml = xml_valid(svg_text)
        unconditional_samples.append({'svg': str(svg_path), 'png': str(png_path), 'temperature': temp, 'xml_valid': valid_xml, 'rendered': rendered})
        print(f'Uncond {idx+1} temp={temp:.1f} xml_valid={valid_xml} rendered={rendered}')

    prefix_examples = [
        {
            'name': 'partial_face',
            'prefix': '<svg viewBox="0 0 64 64" xmlns="http://www.w3.org/2000/svg"><circle cx="20" cy="24" r="8" stroke="black" fill="none"/><circle cx="44" cy="24" r="4" fill="black"/>'
        },
        {
            'name': 'open_path',
            'prefix': '<svg viewBox="0 0 64 64" xmlns="http://www.w3.org/2000/svg"><path d="M10 10 L54 10 L54 54" stroke="black" fill="none"'
        },
        {
            'name': 'one_shape_group',
            'prefix': '<svg viewBox="0 0 64 64" xmlns="http://www.w3.org/2000/svg"><g><rect x="12" y="12" width="40" height="20" fill="#1f77b4"/>'
        },
        {
            'name': 'circle_with_slice',
            'prefix': '<svg viewBox="0 0 64 64" xmlns="http://www.w3.org/2000/svg"><circle cx="32" cy="32" r="18" fill="none" stroke="black"/><path d="M32 14 L32 50" stroke="black"/>'
        },
        {
            'name': 'half_icon',
            'prefix': '<svg viewBox="0 0 64 64" xmlns="http://www.w3.org/2000/svg"><rect x="10" y="18" width="44" height="28" rx="6" fill="none" stroke="black"/>'
        },
    ]

    prefix_results = []
    print('Generating prefix-conditioned samples...')
    for example in prefix_examples:
        for temp in args.sample_temps:
            generated_ids = sample_sequence(
                model,
                tokenizer,
                safe_encode(tokenizer, example['prefix']),
                max_new_tokens=args.max_gen_tokens,
                device=device,
                temperature=temp,
                top_k=args.top_k,
                top_p=args.top_p,
            )
            svg_text = decode_tokens(tokenizer, generated_ids)
            svg_name = f"prefix_{example['name']}_t{temp:.1f}.svg"
            svg_path = SAMPLES_DIR / svg_name
            svg_path.write_text(svg_text, encoding='utf-8')
            png_path = RENDER_DIR / svg_name.replace('.svg', '.png')
            rendered = render_svg(svg_text, png_path)
            valid_xml = xml_valid(svg_text)
            prefix_results.append({
                'name': example['name'],
                'prefix': example['prefix'],
                'svg': str(svg_path),
                'png': str(png_path),
                'temperature': temp,
                'xml_valid': valid_xml,
                'rendered': rendered,
            })
            print(f"Prefix {example['name']} temp={temp:.1f} xml_valid={valid_xml} rendered={rendered}")

    sample_pngs = [r['png'] for r in unconditional_samples[:8]] + [r['png'] for r in prefix_results[:8]]
    sample_pngs = [p for p in sample_pngs if Path(p).exists()]
    if sample_pngs:
        grid_path = OUTPUT_DIR / 'sample_grid.png'
        plot_sample_grid(sample_pngs, grid_path)
        print('Saved sample grid to', grid_path)

    metrics = {
        'model': args.model,
        'params_m': sum(p.numel() for p in model.parameters()) / 1e6,
        'val_loss': eval_loss,
        'val_perplexity': eval_perplexity,
        'test_loss': test_loss,
        'test_perplexity': test_perplexity,
        'unconditional_samples': unconditional_samples,
        'prefix_samples': prefix_results,
        'training': {
            'epochs': args.epochs,
            'batch_size': args.batch_size,
            'block_size': args.block_size,
            'learning_rate': lr,
            'weight_decay': args.weight_decay,
        },
        'sampling': {
            'temperatures': args.sample_temps,
            'top_k': args.top_k,
            'top_p': args.top_p,
        },
    }
    with open(METRICS_PATH, 'w', encoding='utf-8') as f:
        json.dump(metrics, f, indent=2)
    print('Saved metrics to', METRICS_PATH)


from typing import Optional

def run_part4(config: Optional[Config] = None):
    config = config or Config()
    main(config)

if __name__ == '__main__':
    run_part4()


Auto-selected best model from Part 2/Part 3 results: large
Model configuration: large {'d_model': 512, 'n_layers': 6, 'n_heads': 16, 'd_ff': 2048}
Total parameters: 25.07M
Using learning rate: 0.0006
Training for 3 epochs, 7348 steps per epoch...
Step 50/7348 avg_loss=7.4409 lr=2.04226e-05
Step 100/7348 avg_loss=3.9973 lr=4.08447e-05
Step 150/7348 avg_loss=2.3470 lr=6.12667e-05
Step 200/7348 avg_loss=1.9024 lr=8.16887e-05
Step 250/7348 avg_loss=1.7697 lr=0.000102111
Step 300/7348 avg_loss=1.6913 lr=0.000122533
Step 350/7348 avg_loss=1.6539 lr=0.000142955
Step 400/7348 avg_loss=1.5688 lr=0.000163377
Step 450/7348 avg_loss=1.5182 lr=0.000183799
Step 500/7348 avg_loss=1.4352 lr=0.000204221
Step 550/7348 avg_loss=1.3799 lr=0.000224643
Step 600/7348 avg_loss=1.2918 lr=0.000245065
Step 650/7348 avg_loss=1.2641 lr=0.000265487
Step 700/7348 avg_loss=1.2220 lr=0.000285909
Step 750/7348 avg_loss=1.1799 lr=0.000306331
Step 800/7348 avg_loss=1.1610 lr=0.000326753
Step 850/7348 avg_loss=1.1335 lr=0

/home/av4008/.local/lib/python3.9/site-packages/torch/optim/lr_scheduler.py:198: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


Step 1500/7348 avg_loss=0.8782 lr=0.000599997
Step 1550/7348 avg_loss=0.8412 lr=0.000599977
Step 1600/7348 avg_loss=0.8398 lr=0.00059994
Step 1650/7348 avg_loss=0.8451 lr=0.000599885
Step 1700/7348 avg_loss=0.8372 lr=0.000599813
Step 1750/7348 avg_loss=0.8180 lr=0.000599724
Step 1800/7348 avg_loss=0.8240 lr=0.000599617
Step 1850/7348 avg_loss=0.8113 lr=0.000599492
Step 1900/7348 avg_loss=0.7994 lr=0.000599351
Step 1950/7348 avg_loss=0.7906 lr=0.000599191
Step 2000/7348 avg_loss=0.7801 lr=0.000599014
Step 2050/7348 avg_loss=0.7742 lr=0.00059882
Step 2100/7348 avg_loss=0.7816 lr=0.000598609
Step 2150/7348 avg_loss=0.7678 lr=0.00059838
Step 2200/7348 avg_loss=0.7637 lr=0.000598133
Step 2250/7348 avg_loss=0.7642 lr=0.000597869
Step 2300/7348 avg_loss=0.7661 lr=0.000597588
Step 2350/7348 avg_loss=0.7787 lr=0.00059729
Step 2400/7348 avg_loss=0.8011 lr=0.000596974
Step 2450/7348 avg_loss=0.7602 lr=0.000596641
Step 2500/7348 avg_loss=0.7469 lr=0.00059629
Step 2550/7348 avg_loss=0.7407 lr=0.000

# Testing 

In [ ]:
!pip install tokenizers


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import torch
from tokenizers import Tokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'

TOKENIZER_PATH = 'svg_tokenizer_6.json'
MODEL_PATH = 'part4_outputs/best_model.pt'

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

model = DecoderOnlyTransformer(
    vocab_size=tokenizer.get_vocab_size(),
    d_model=512,
    n_layers=6,
    n_heads=16,
    d_ff=2048,
    max_len=512,   # 🔥 CRITICAL FIX
).to(device)

state = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(state)

model.eval()
print("Model loaded correctly.")

Model loaded correctly.


In [ ]:
from IPython.display import display, SVG
from lxml import etree

def clean_svg(text):
    text = text.replace("Ġ", " ")
    text = text.replace("< svg", "<svg")
    text = text.replace("</ svg", "</svg")
    text = text.replace(" >", ">")
    return text.strip()

def fix_svg(svg):
    if "<svg" not in svg:
        svg = "<svg>" + svg
    if "</svg>" not in svg:
        svg += "</svg>"
    return svg

def render_svg_notebook(svg_text):
    try:
        display(SVG(svg_text))
        return True
    except Exception:
        return False

def xml_valid(svg_text):
    try:
        etree.fromstring(svg_text.encode('utf-8'))
        return True
    except:
        return False

In [ ]:
def generate_svg(prefix, temp=0.8):
    ids = sample_sequence(
        model,
        tokenizer,
        safe_encode(tokenizer, prefix),
        max_new_tokens=256,
        device=device,
        temperature=temp,
        top_k=40,
        top_p=0.9,
    )
    
    text = decode_tokens(tokenizer, ids)
    text = clean_svg(text)
    text = fix_svg(text)
    
    return text

In [ ]:
print("=== Unconditional Samples ===")

temps = [0.5, 0.8, 1.0]

for i in range(10):
    temp = temps[i % len(temps)]
    
    svg = generate_svg("<svg ", temp=temp)
    
    print(f"\nSample {i+1} | temp={temp}")
    print("XML valid:", xml_valid(svg))
    
    render_svg_notebook(svg)

=== Unconditional Samples ===

Sample 1 | temp=0.5
XML valid: False

Sample 2 | temp=0.8
XML valid: False

Sample 3 | temp=1.0
XML valid: False

Sample 4 | temp=0.5
XML valid: False

Sample 5 | temp=0.8
XML valid: False

Sample 6 | temp=1.0
XML valid: False

Sample 7 | temp=0.5
XML valid: False

Sample 8 | temp=0.8
XML valid: False

Sample 9 | temp=1.0
XML valid: False

Sample 10 | temp=0.5
XML valid: False


In [ ]:
prefixes = [
    '<svg viewBox="0 0 64 64"><circle cx="20" cy="24" r="8"/>',
    '<svg viewBox="0 0 64 64"><path d="M10 10 L54 10 L54 54"',
    '<svg viewBox="0 0 64 64"><g><rect x="12" y="12" width="40" height="20"/>',
    '<svg viewBox="0 0 64 64"><circle cx="32" cy="32" r="18"/>',
    '<svg viewBox="0 0 64 64"><rect x="10" y="18" width="44" height="28"/>',
]

In [ ]:
print("=== Prefix-conditioned Samples ===")

for prefix in prefixes:
    for temp in [0.5, 0.8, 1.0]:
        
        svg = generate_svg(prefix, temp=temp)
        
        print("\n---")
        print("Temp:", temp)
        print("XML valid:", xml_valid(svg))
        
        print("Prefix:")
        print(prefix)
        
        render_svg_notebook(svg)

=== Prefix-conditioned Samples ===

---
Temp: 0.5
XML valid: False
Prefix:
<svg viewBox="0 0 64 64"><circle cx="20" cy="24" r="8"/>

---
Temp: 0.8
XML valid: False
Prefix:
<svg viewBox="0 0 64 64"><circle cx="20" cy="24" r="8"/>

---
Temp: 1.0
XML valid: False
Prefix:
<svg viewBox="0 0 64 64"><circle cx="20" cy="24" r="8"/>

---
Temp: 0.5
XML valid: False
Prefix:
<svg viewBox="0 0 64 64"><path d="M10 10 L54 10 L54 54"

---
Temp: 0.8
XML valid: False
Prefix:
<svg viewBox="0 0 64 64"><path d="M10 10 L54 10 L54 54"

---
Temp: 1.0
XML valid: False
Prefix:
<svg viewBox="0 0 64 64"><path d="M10 10 L54 10 L54 54"

---
Temp: 0.5
XML valid: False
Prefix:
<svg viewBox="0 0 64 64"><g><rect x="12" y="12" width="40" height="20"/>

---
Temp: 0.8
XML valid: False
Prefix:
<svg viewBox="0 0 64 64"><g><rect x="12" y="12" width="40" height="20"/>

---
Temp: 1.0
XML valid: False
Prefix:
<svg viewBox="0 0 64 64"><g><rect x="12" y="12" width="40" height="20"/>

---
Temp: 0.5
XML valid: False
Prefix:
<svg vi

In [ ]:
import re

def sanitize_svg(text: str) -> str:
    # remove tokenizer artifacts
    text = text.replace("Ġ", " ")
    text = text.replace("< svg", "<svg")
    text = text.replace("</ svg", "</svg")

    # remove illegal characters before first svg
    text = re.sub(r"^[^<]*", "", text)

    # force svg root
    if "<svg" not in text:
        text = "<svg>" + text

    # cut everything after </svg> if possible
    if "</svg>" in text:
        text = text.split("</svg>")[0] + "</svg>"
    else:
        text += "</svg>"

    return text.strip()
from IPython.display import SVG, display
from lxml import etree

def is_valid(svg):
    try:
        etree.fromstring(svg.encode("utf-8"))
        return True
    except:
        return False


def safe_show(svg_text):
    svg_text = sanitize_svg(svg_text)

    print("VALID XML:", is_valid(svg_text))

    try:
        display(SVG(svg_text))
    except Exception as e:
        print("Render failed — showing raw text instead")
        print(svg_text[:500])
def generate_svg(model, tokenizer, prefix, temp=0.8):
    ids = sample_sequence(
        model,
        tokenizer,
        safe_encode(tokenizer, prefix),
        max_new_tokens=256,
        device=device,
        temperature=temp,
        top_k=40,
        top_p=0.9,
    )

    raw = decode_tokens(tokenizer, ids)
    return sanitize_svg(raw)


In [ ]:
for i in range(5):
    svg = generate_svg(model, tokenizer, "<svg ", temp=0.8)

    print("\n--- SAMPLE", i, "---")
    safe_show(svg)


--- SAMPLE 0 ---
VALID XML: False
Render failed — showing raw text instead
<svg   . 9  14 . 8  14 . 8  14 . 8  14 . 7  L 14 . 8  13 . 4  L 15 . 5  13 . 4  L 15 . 5  14 . 7  C 15 . 5  14 . 8  15 . 5  14 . 9  15 . 5  15 . 0  L 14 . 8  15 . 0  L 14 . 8  17 . 8  C 14 . 7  18 . 1  14 . 5  18 . 3  14 . 4  18 . 5  L 9 . 8  18 . 5  C 9 . 5  18 . 3  9 . 3  18 . 1  9 . 1  17 . 8  L 9 . 1  15 . 0  L 8 . 5  15 . 0  C 8 . 4  14 . 9  8 . 2  14 . 8  8 . 0  14 . 7  L 8 . 0  13 . 4  L 8 . 5  13 . 4  L 8 . 5  12 . 5  L 9 . 1  12 . 5  C 9 . 3  12 . 4  9 . 5  12 . 3  9 . 7  12 . 1  L 9 . 

--- SAMPLE 1 ---
VALID XML: False
Render failed — showing raw text instead
<svg   . 2  17 . 4  L 3 . 9  17 . 4  C 2 . 1  17 . 2  1 . 7  16 . 6  1 . 2  15 . 8  L 1 . 2  10 . 4  C 1 . 7  9 . 4  2 . 1  8 . 5  3 . 9  8 . 2  L 3 . 9  8 . 2 "></ path >  < path  fill =" none "  stroke =" black "  stroke - width =". 3 "  stroke - opacity =" 1 . 0 "  filling =" 0 "  d =" M 3 . 9  8 . 2  L 3 . 9  8 . 2  C 4 . 2  8 . 4  4 . 5  8 

In [18]:
import re
from IPython.display import SVG, display

def clean_svg(text: str) -> str:
    # fix spaced decimals: "14 . 8" → "14.8"
    text = re.sub(r"(\d)\s*\.\s*(\d)", r"\1.\2", text)
    text=re.sub("Ġ", " ", text)

    # fix broken tag spacing: "< path >" → "<path>"
    text = re.sub(r"<\s*/\s*", "</", text)
    text = re.sub(r"<\s*", "<", text)
    text = re.sub(r"\s*>", ">", text)

    # fix attribute splits
    text = re.sub(r"stroke\s*-\s*width", "stroke-width", text)
    text = re.sub(r"stroke\s*-\s*opacity", "stroke-opacity", text)
    text = re.sub(r"fill\s*=\s*\"?\s*none\s*\"?", "fill=\"none\"", text)

    # remove stray isolated dots
    text = re.sub(r"\s\.\s", " ", text)

    # ensure closing svg tag
    if "<svg" in text and "</svg>" not in text:
        text += "\n</svg>"

    return text


def show_generation(model, tokenizer, prefix="<svg ", temp=0.8, max_new_tokens=200):
    ids = sample_sequence(
        model,
        tokenizer,
        safe_encode(tokenizer, prefix),
        max_new_tokens=max_new_tokens,
        device=device,
        temperature=temp,
        top_k=5,
        top_p=0.9,
    )

    raw_svg = decode_tokens(tokenizer, ids)
    clean = clean_svg(raw_svg)

    print("\n--- VALID XML CHECK ---")
    print("Raw:", xml_valid(raw_svg))
    print("Clean:", xml_valid(clean))

    print("\n--- DISPLAY ---")
    try:
        
        display(SVG(clean))
    except Exception as e:
        print("Render failed, showing text instead:\n")
        print(clean[:2000])

    return clean


# # RUN TEST (edit temp if needed)
# for i in range(3):
#     print(f"\n\n========== SAMPLE {i} ==========")
#     show_generation(model, tokenizer, "<svg xmlns='http://www.w3.org/2000/svg'>", temp=0.8)

In [20]:
# Complete the partial SVG using the model
svg_prefix = '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.0 24.0 24.0" height="200px" width="200px"><path fill="none" stroke="black" stroke-width=".3" stroke-opacity="1.0" filling="0" d="M10.199999809265137 2.1000003814697266 L12.0 2.1000003814697266 L12.0 8.700000762939453 L12.0 15.300000190734863 L12.0 21.899999618530273 L10.199999809265137 21.899999618530273 L10.199999809265137 20.099998474121094 L5.700000286102295 20.099998474121094 C4.916017532348633 19.775388717651367 4.224609851837158 19.083980560302734 3.90000057220459 18.299999237060547 L3.90000057220459 12.0 L3.90000057220459 5.700000286102295 C4.224610328674316 4.916018009185791 4.916018009185791 4.224610328674316 5.700000286102295 3.90000057220459 L10.199999809265137 3.90000057220459 L10.199999809265137 2.1000003814697266">'
print("SVG Prefix:")
print(svg_prefix)
print("\n" + "="*60)
print("Generating SVG completion with temperature=0.7...")
print("="*60)

# Use the model to complete the SVG
completed_svg = show_generation(model, tokenizer, prefix=svg_prefix, temp=0.3, max_new_tokens=256)

print("\n" + "="*60)
print("Completed SVG:")
print("="*60)
print(completed_svg)

SVG Prefix:
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.0 24.0 24.0" height="200px" width="200px"><path fill="none" stroke="black" stroke-width=".3" stroke-opacity="1.0" filling="0" d="M10.199999809265137 2.1000003814697266 L12.0 2.1000003814697266 L12.0 8.700000762939453 L12.0 15.300000190734863 L12.0 21.899999618530273 L10.199999809265137 21.899999618530273 L10.199999809265137 20.099998474121094 L5.700000286102295 20.099998474121094 C4.916017532348633 19.775388717651367 4.224609851837158 19.083980560302734 3.90000057220459 18.299999237060547 L3.90000057220459 12.0 L3.90000057220459 5.700000286102295 C4.224610328674316 4.916018009185791 4.916018009185791 4.224610328674316 5.700000286102295 3.90000057220459 L10.199999809265137 3.90000057220459 L10.199999809265137 2.1000003814697266">

Generating SVG completion with temperature=0.7...

--- VALID XML CHECK ---
Raw: False
Clean: False

--- DISPLAY ---
Render failed, showing text instead:

 <svg  xmlns =" http :// www w 3 org / 

In [17]:
# QUICK GENERATION - Load model and generate samples WITHOUT retraining
import torch
from tokenizers import Tokenizer
from IPython.display import display, SVG
from lxml import etree
import re

# === Setup ===
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

# Load tokenizer and model
tokenizer = Tokenizer.from_file('svg_tokenizer_6.json')
model = DecoderOnlyTransformer(
    vocab_size=tokenizer.get_vocab_size(),
    d_model=512,
    n_layers=6,
    n_heads=16,
    d_ff=2048,
    max_len=512,
).to(device)

# Load pretrained weights
state = torch.load('part4_outputs/best_model.pt', map_location=device)
model.load_state_dict(state)
model.eval()
print("✓ Model loaded successfully\n")

# === Generate samples with DEBUG output ===
print("="*70)
print("GENERATING SAMPLES WITH DEBUGGING")
print("="*70)

for temp in [0.2, 0.35]:
    print(f"\n{'='*70}")
    print(f"Temperature: {temp}")
    print('='*70)
    
    ids = sample_sequence_improved(
        model,
        tokenizer,
        safe_encode(tokenizer, '<svg '),
        max_new_tokens=200,
        device=device,
        temperature=temp,
        top_k=50,
        top_p=0.95,
    )
    
    raw_svg = decode_tokens(tokenizer, ids)
    
    print(f"\n--- RAW OUTPUT (first 500 chars) ---")
    print(raw_svg[:500])
    
    svg_text = repair_svg(raw_svg)
    
    print(f"\n--- REPAIRED (first 500 chars) ---")
    print(svg_text[:500])
    
    valid = is_valid_svg(svg_text)
    print(f"\n--- VALIDATION ---")
    print(f"Valid XML: {valid}")
    
    if not valid:
        print(f"Full repaired SVG:")
        print(svg_text)
        
        # Try to see the parse error
        try:
            etree.fromstring(svg_text.encode('utf-8'))
        except Exception as e:
            print(f"\nParse error: {e}")
    
    if valid:
        try:
            display(SVG(svg_text))
        except Exception as e:
            print(f"Render error: {e}")


Device: cpu
✓ Model loaded successfully

GENERATING SAMPLES WITH DEBUGGING

Temperature: 0.2

--- RAW OUTPUT (first 500 chars) ---
Ġ< svg Ġ . 0 ĠL 8 . 4 Ġ4 . 8 ĠL 15 . 6 Ġ4 . 8 ĠL 22 . 8 Ġ4 . 8 ĠL 22 . 8 Ġ12 . 0 ĠL 22 . 8 Ġ19 . 2 ĠL 15 . 6 Ġ19 . 2 ĠL 8 . 4 Ġ19 . 2 ĠL 1 . 2 Ġ19 . 2 ĠL 1 . 2 Ġ12 . 0 ĠL 1 . 2 Ġ4 . 8 "></ path > Ġ< path Ġfill =" none " Ġstroke =" black " Ġstroke - width =". 3 " Ġstroke - opacity =" 1 . 0 " Ġfilling =" 0 " Ġd =" M 2 . 6 Ġ5 . 7 ĠL 8 . 9 Ġ5 . 7 ĠL 15 . 2 Ġ5 . 7 ĠL 21 . 5 Ġ5 . 7 ĠL 21 . 5 Ġ12 . 0 ĠL 21 . 5 Ġ18 . 3 ĠL 15 . 2 Ġ18 . 3 ĠL 8 . 9 Ġ18 . 3 ĠL 2 . 6 Ġ18 . 3 ĠL 2 . 6 Ġ12 . 0 ĠL 2 . 6 Ġ5 . 7 

--- REPAIRED (first 500 chars) ---
<svg . 0 L 8.4 4.8 L 15.6 4.8 L 22.8 4.8 L 22.8 12.0 L 22.8 19.2 L 15.6 19.2 L 8.4 19.2 L 1.2 19.2 L 1.2 12.0 L 1.2 4.8 "></path> <path fill="none" stroke =" black " stroke-width =". 3 " stroke-opacity =" 1.0 " filling =" 0 " d =" M 2.6 5.7 L 8.9 5.7 L 15.2 5.7 L 21.5 5.7 L 21.5 12.0 L 21.5 18.3 L 15.2 18.3 L 8.9 18.3 L 2.6 18.3 L

In [ ]:
import re
from IPython.display import display, SVG

def repair_svg(text: str) -> str:
    """Aggressive SVG repair to fix generation artifacts."""
    if not text:
        return '<svg viewBox="0 0 100 100"></svg>'
    
    # Step 0: Remove everything before first <
    match = re.search(r'<', text)
    if match:
        text = text[match.start():]
    else:
        return '<svg viewBox="0 0 100 100"></svg>'
    
    # Step 1: Remove tokenizer artifacts
    text = text.replace("Ġ", " ")
    text = text.replace("ĠĠ", " ")
    text = text.replace("Ċ", "\n")
    text = re.sub(r'\s+', ' ', text)
    
    # Step 2: Fix broken tag spacing AGGRESSIVELY
    text = re.sub(r'<\s+', '<', text)
    text = re.sub(r'\s+>', '>', text)
    text = re.sub(r'<\s*/\s*', '</', text)
    text = re.sub(r'\s*=\s*', '=', text)  # Remove spaces around =
    
    # Step 3: Fix broken attribute names (with spaces around hyphens)
    attr_fixes = [
        ('stroke-width', 'stroke-width'),
        ('stroke-linecap', 'stroke-linecap'),
        ('stroke-linejoin', 'stroke-linejoin'),
        ('stroke-dasharray', 'stroke-dasharray'),
        ('stroke-opacity', 'stroke-opacity'),
        ('fill-opacity', 'fill-opacity'),
        ('font-size', 'font-size'),
        ('font-family', 'font-family'),
        ('font-weight', 'font-weight'),
        ('text-anchor', 'text-anchor'),
    ]
    for correct in attr_fixes:
        # Fix patterns like "stroke - width" or "stroke  -  width"
        broken = correct.replace('-', r'\s*-\s*')
        text = re.sub(broken, correct, text, flags=re.IGNORECASE)
    
    # Step 4: Fix broken numbers
    text = re.sub(r'(\d)\s+\.(\d)', r'\1.\2', text)  # "14 .8" -> "14.8"
    text = re.sub(r'(\d)\s+e\s*-\s*(\d)', r'\1e-\2', text)  # Scientific notation
    
    # Step 5: Ensure quotes on attribute values (aggressive)
    text = re.sub(r'=([a-zA-Z0-9#.]+)(?=[>\s])', r'="\1"', text)
    
    # Step 6: Cut everything after first closing svg
    if '</svg>' in text.lower():
        idx = text.lower().find('</svg>') + 6
        text = text[:idx]
    
    # Step 7: Ensure SVG root exists and is open
    if '<svg' not in text.lower():
        text = '<svg viewBox="0 0 100 100">' + text
    
    # Step 8: Close unclosed common tags
    unclosed_tags = ['g', 'path', 'rect', 'circle', 'ellipse', 'text', 'tspan', 
                     'line', 'polyline', 'polygon', 'defs', 'style', 'use', 'image']
    
    for tag in unclosed_tags:
        # Count opening and closing tags (case-insensitive)
        open_count = len(re.findall(rf'<{tag}(?:\s|>|/)', text, re.IGNORECASE))
        close_count = len(re.findall(rf'</{tag}>', text, re.IGNORECASE))
        
        # Close extra open tags
        if open_count > close_count:
            for _ in range(open_count - close_count):
                text += f'</{tag}>'
    
    # Step 9: Ensure closing SVG tag exists
    if '</svg>' not in text.lower():
        text += '</svg>'
    
    # Step 10: Remove any duplicate closing tags
    text = re.sub(r'(</[^>]+>)\s*\1+', r'\1', text, flags=re.IGNORECASE)
    
    return text.strip()


def is_valid_svg(svg_text: str) -> bool:
    """Check if SVG is valid XML."""
    if not svg_text or len(svg_text.strip()) < 5:
        return False
    try:
        root = etree.fromstring(svg_text.encode('utf-8'))
        # Make sure it's actually SVG
        tag = root.tag.lower()
        return 'svg' in tag
    except Exception as e:
        return False


def sample_sequence_improved(model, tokenizer, prefix_ids, max_new_tokens, device, 
                            temperature=0.3, top_k=50, top_p=0.95, eos_token_id=None):
    """Improved sampling that stops at closing SVG tag."""
    model.eval()
    generated = prefix_ids[:]
    
    with torch.no_grad():
        for step in range(max_new_tokens):
            context = torch.tensor([generated[-512:]], dtype=torch.long, device=device)
            logits = model(context)
            next_token_logits = logits[0, -1, :] / max(1e-8, temperature)
            next_token_logits = top_k_top_p_filtering(next_token_logits, top_k=top_k, top_p=top_p)
            probs = torch.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()
            generated.append(next_token)
            
            # Check for stopping tokens
            if eos_token_id is not None and next_token == eos_token_id:
                break
            
            # Try to detect </svg> and stop early
            try:
                current_text = decode_tokens(tokenizer, generated).lower()
                if '</svg>' in current_text:
                    break
            except Exception:
                pass
    
    return generated
